In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import warnings
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

DATA_PATH   = '../../../data/alternative/student_modeling_table.csv'
OUTPUT_DIR  = '../../../output/alternative_model'
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEED = 42

df = pd.read_csv(DATA_PATH)
print(f'Loaded: {df.shape}')
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: '../../../data/alternative/student_modeling_table.csv'

In [ ]:
# user_id is an arbitrary UUID — no predictive signal
df = df.drop(columns=['user_id'])
print(f'After dropping user_id: {df.shape}')

After dropping user_id: (10000, 22)


In [ ]:
# Ordinal encoding (keeps natural ordering where it exists)
program_map    = {'college': 0, 'university': 1}
living_map     = {'family': 0, 'dorm': 1, 'rent': 2}   # family = cheapest / most stable
potential_map  = {'low': 0, 'medium': 1, 'high': 2}

df['program_level']          = df['program_level'].map(program_map)
df['living_status']          = df['living_status'].map(living_map)
df['major_income_potential'] = df['major_income_potential'].map(potential_map)

print('Encoding complete.')
print(df[['program_level', 'living_status', 'major_income_potential']].value_counts().head(10))

Encoding complete.
program_level  living_status  major_income_potential
1              0              0                         1391
                              1                         1083
               1              0                         1048
                              1                          896
0              0              0                          805
1              2              0                          678
0              1              0                          652
1              2              1                          560
               0              2                          537
               1              2                          446
Name: count, dtype: int64


In [ ]:
# --- ENHANCED FEATURE ENGINEERING ---
# Deriving new compound signals to help XGBoost delineate risk

# 1. Financial Stress Index
df['financial_stress_index'] = df['debt_ratio'] * df['behavior_volatility']

# 2. Academic Resilience (Counteracts bad grades with strong support)
df['academic_resilience'] = df['gpa_latest'] * df['support_numeric']

# 3. Risk Compounding (Adds up dangerous categorical flags)
df['risk_compounding'] = df['severe_behavior_flag'] + df['thin_support_flag'] + df['high_pressure_flag']

# 4. Loan Burden to Maturity
df['loan_to_maturity_ratio'] = df['loan_amount'] / (df['maturity_score'] + 0.1)

print('Added new synthetic interaction features.')

Added new synthetic interaction features.


In [ ]:
# Define final feature set (all columns except target)
FEATURE_COLS = [col for col in df.columns if col != 'default']
print(f'Total features: {len(FEATURE_COLS)}')
print(FEATURE_COLS)

Total features: 25
['age', 'program_level', 'living_status', 'academic_year', 'maturity_score', 'gpa_latest', 'major_income_potential', 'loan_amount', 'debt_ratio', 'high_pressure_flag', 'behavior_risk_score', 'behavior_volatility', 'severe_behavior_flag', 'support_numeric', 'has_buffer', 'thin_support_flag', 'debt_x_behavior', 'debt_x_support', 'debt_x_living', 'behavior_under_pressure', 'shock_vulnerability', 'financial_stress_index', 'academic_resilience', 'risk_compounding', 'loan_to_maturity_ratio']


In [ ]:
X = df[FEATURE_COLS]
y = df['default']

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_SEED, stratify=y
)

# Second split: temp → 50% val, 50% test (= 15% each of total)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_SEED, stratify=y_temp
)

print('Split sizes:')
print(f'  Train : {len(X_train):>5,} rows ({len(X_train)/len(X)*100:.1f}%)')
print(f'  Val   : {len(X_val):>5,} rows ({len(X_val)/len(X)*100:.1f}%)')
print(f'  Test  : {len(X_test):>5,} rows ({len(X_test)/len(X)*100:.1f}%)')

print('\nClass distribution (default rate):')
for name, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    print(f'  {name}: {y_split.mean()*100:.2f}%')

Split sizes:
  Train : 7,000 rows (70.0%)
  Val   : 1,500 rows (15.0%)
  Test  : 1,500 rows (15.0%)

Class distribution (default rate):
  Train: 35.79%
  Val: 35.80%
  Test: 35.73%


In [ ]:
# Raw splits (for tree models — LightGBM / XGBoost)
X_train.to_csv(f'{OUTPUT_DIR}/X_train.csv', index=False)
X_val.to_csv(f'{OUTPUT_DIR}/X_val.csv',     index=False)
X_test.to_csv(f'{OUTPUT_DIR}/X_test.csv',   index=False)
y_train.to_csv(f'{OUTPUT_DIR}/y_train.csv', index=False)
y_val.to_csv(f'{OUTPUT_DIR}/y_val.csv',     index=False)
y_test.to_csv(f'{OUTPUT_DIR}/y_test.csv',   index=False)

# Feature list
with open(f'{OUTPUT_DIR}/feature_cols.pkl', 'wb') as f:
    pickle.dump(FEATURE_COLS, f)

print('All outputs saved to:', OUTPUT_DIR)
print('Files:', os.listdir(OUTPUT_DIR))

All outputs saved to: ../../../output/alternative_model
Files: ['best_model_xgboost.pkl', 'feature_cols.pkl', 'model_comparison.csv', 'scaler.pkl', 'shap_feature_importance.png', 'X_test.csv', 'X_test_scaled.csv', 'X_train.csv', 'X_train_scaled.csv', 'X_val.csv', 'X_val_scaled.csv', 'y_test.csv', 'y_train.csv', 'y_val.csv']
